In [1]:
import pandas as pd

# Absolute path — works no matter where the notebook file is saved
Data_path = r"D:\Python Project\marketpulseAi\data\raw\\"

ecommerce = {
    "customers": pd.read_csv(Data_path + "olist_customers_dataset.csv"),
    "orders": pd.read_csv(Data_path + "olist_orders_dataset.csv"),
    "order_items": pd.read_csv(Data_path + "olist_order_items_dataset.csv"),
    "payments": pd.read_csv(Data_path + "olist_order_payments_dataset.csv"),
    "review": pd.read_csv(Data_path + "olist_order_reviews_dataset.csv"),
    "products": pd.read_csv(Data_path + "olist_products_dataset.csv"),
    "sellers": pd.read_csv(Data_path + "olist_sellers_dataset.csv"),
    "geolocation": pd.read_csv(Data_path + "olist_geolocation_dataset.csv"),
    "category_transalation": pd.read_csv(Data_path + "product_category_name_translation.csv"),
}

# Sanity check: print shape of every table
for name, df in ecommerce.items():
    print(f"{name:25s} → {df.shape[0]:>7,} rows | {df.shape[1]} columns")

customers                 →  99,441 rows | 5 columns
orders                    →  99,441 rows | 8 columns
order_items               → 112,650 rows | 7 columns
payments                  → 103,886 rows | 5 columns
review                    →  99,224 rows | 7 columns
products                  →  32,951 rows | 9 columns
sellers                   →   3,095 rows | 4 columns
geolocation               → 1,000,163 rows | 5 columns
category_transalation     →      71 rows | 2 columns


In [14]:
#Clean And Aggregate Geolocation Table
geo=ecommerce["geolocation"]
print(f"Before dedup: {len(geo):,} rows")
#remove Duplicate row
geo_dedup=geo.drop_duplicates()
lat_ok=geo_dedup["geolocation_lat"].between(-34,5)
lng_ok=geo_dedup["geolocation_lng"].between(-74,-34)
#filter out coordinates outside Brazil's real bounds
geo_valid=geo_dedup[lat_ok & lng_ok]
print(f"After removing out-of-bounds rows: {len(geo_valid):,} rows")
#Aggregate to one row per zip prefix
geo_clean=geo_valid.groupby("geolocation_zip_code_prefix").agg(
geolocation_lat=("geolocation_lat", "mean"),
    geolocation_lng=("geolocation_lng", "mean"),
    geolocation_city=("geolocation_city", lambda x: x.mode()[0]),
    geolocation_state=("geolocation_state", lambda x: x.mode()[0]),
).reset_index()
print(f"\nFinal geo_clean shape: {geo_clean.shape}")
print(f"Duplicate zip prefixes remaining: {geo_clean['geolocation_zip_code_prefix'].duplicated().sum()}")
print(geo_clean.head())
print(f"geo_dedup rows: {len(geo_dedup):,}")
print(f"Difference from original: {len(geo) - len(geo_dedup):,}")    


Before dedup: 1,000,163 rows
After removing out-of-bounds rows: 738,299 rows

Final geo_clean shape: (19010, 5)
Duplicate zip prefixes remaining: 0
   geolocation_zip_code_prefix  geolocation_lat  geolocation_lng  \
0                         1001       -23.550227       -46.634039   
1                         1002       -23.547657       -46.634991   
2                         1003       -23.549000       -46.635582   
3                         1004       -23.549829       -46.634792   
4                         1005       -23.549547       -46.636406   

  geolocation_city geolocation_state  
0        sao paulo                SP  
1        sao paulo                SP  
2        sao paulo                SP  
3        sao paulo                SP  
4        sao paulo                SP  
geo_dedup rows: 738,332
Difference from original: 261,831


In [2]:
orders = ecommerce["orders"]

# Recompute the same three conditions from Phase 1 Module 3
shipped_before_approved = orders["order_delivered_carrier_date"] < orders["order_approved_at"]
delivered_before_shipped = orders["order_delivered_customer_date"] < orders["order_delivered_carrier_date"]

# Combine into a single flag column.
# fillna(False): if either date is missing (NaT), the comparison is already False,
# but being explicit here makes the intent clear.
orders["has_timeline_violation"] = (
    shipped_before_approved.fillna(False) | delivered_before_shipped.fillna(False)
)

print(f"Rows flagged: {orders['has_timeline_violation'].sum():,}")
print(f"Total orders: {len(orders):,}")
print(f"Flagged %: {orders['has_timeline_violation'].mean() * 100:.2f}%")

# Quick sanity check: breakdown by order_status for flagged rows
print("\nStatus breakdown of flagged rows:")
print(orders.loc[orders["has_timeline_violation"], "order_status"].value_counts())

Rows flagged: 1,382
Total orders: 99,441
Flagged %: 1.39%

Status breakdown of flagged rows:
order_status
delivered    1373
shipped         9
Name: count, dtype: int64


In [5]:
products = ecommerce["products"]
category_translation = ecommerce["category_transalation"]

# --- Fix 1: Add missing translations ---
missing_translations = pd.DataFrame({
    "product_category_name": ["pc_gamer", "portateis_cozinha_e_preparadores_de_alimentos"],
    "product_category_name_english": ["pc_gamer", "kitchen_portables"]
})
category_translation = pd.concat([category_translation, missing_translations], ignore_index=True)
print(f"Translation table now has {len(category_translation)} rows (was 71)")

# --- Fix 2: Label missing product category as 'unknown' ---
before_missing = products["product_category_name"].isna().sum()
products["product_category_name"] = products["product_category_name"].fillna("unknown")
after_missing = products["product_category_name"].isna().sum()
print(f"\nMissing categories before: {before_missing} | after: {after_missing}")

# --- Fix 3: Impute weight=0 with the category's median weight ---
zero_weight_count = (products["product_weight_g"] == 0).sum()
print(f"\nProducts with weight=0 before fix: {zero_weight_count}")

# Replace 0 with NaN first, so groupby median ignores them automatically
products["product_weight_g"] = products["product_weight_g"].replace(0, pd.NA)

# Fill missing weight using each category's own median weight
products["product_weight_g"] = products.groupby("product_category_name")["product_weight_g"] \
    .transform(lambda x: x.fillna(x.median()))

zero_weight_after = (products["product_weight_g"].isna()).sum()
print(f"Products still missing weight after fix: {zero_weight_after}")

# Save back into the dictionary
ecommerce["products"] = products
ecommerce["category_transalation"] = category_translation

Translation table now has 75 rows (was 71)

Missing categories before: 0 | after: 0

Products with weight=0 before fix: 0
Products still missing weight after fix: 0


In [6]:
payments = ecommerce["payments"]

# --- Investigate: are the 3 not_defined rows the full payment record for their order? ---
not_defined_orders = payments.loc[payments["payment_type"] == "not_defined", "order_id"]
print("Orders with not_defined payment type:")
print(payments[payments["order_id"].isin(not_defined_orders)])

# --- Fix: relabel not_defined as "unknown" (clearer semantic meaning, same data) ---
before_count = (payments["payment_type"] == "not_defined").sum()
payments["payment_type"] = payments["payment_type"].replace("not_defined", "unknown")
after_count = (payments["payment_type"] == "not_defined").sum()
print(f"\n'not_defined' rows before: {before_count} | after: {after_count}")

# --- Voucher zero-value rows: confirmed as valid, just documenting, no change needed ---
voucher_zero = payments[(payments["payment_type"] == "voucher") & (payments["payment_value"] == 0)]
print(f"\nVoucher rows with payment_value=0 (kept as-is, valid business case): {len(voucher_zero)}")

# Save back
ecommerce["payments"] = payments


Orders with not_defined payment type:
                               order_id  payment_sequential payment_type  \
51280  4637ca194b6387e2d538dc89b124b0ee                   1  not_defined   
57411  00b1cb0320190ca0daa2c88b35206009                   1  not_defined   
94427  c8c528189310eaa44a745b8d9d26908b                   1  not_defined   

       payment_installments  payment_value  
51280                     1            0.0  
57411                     1            0.0  
94427                     1            0.0  

'not_defined' rows before: 3 | after: 0

Voucher rows with payment_value=0 (kept as-is, valid business case): 6


In [16]:
import os

# ============================================================
# MODULE 4: Payments — fix not_defined, document voucher zeros
# ============================================================
payments = ecommerce["payments"]

# Relabel 'not_defined' -> 'unknown' (clearer meaning, same data, nothing dropped)
before_count = (payments["payment_type"] == "not_defined").sum()
payments["payment_type"] = payments["payment_type"].replace("not_defined", "unknown")
print(f"Module 4: 'not_defined' rows relabeled: {before_count} -> "
      f"{(payments['payment_type'] == 'not_defined').sum()}")

# Voucher zero-value rows are a valid business case (order fully paid by voucher) — kept as-is
voucher_zero = payments[(payments["payment_type"] == "voucher") & (payments["payment_value"] == 0)]
print(f"Voucher rows with payment_value=0 (kept, valid case): {len(voucher_zero)}")

ecommerce["payments"] = payments


# ============================================================
# MODULE 5: Aggregate payments and reviews to one row per order
# (decided in Phase 1 — both tables are 1:N against orders)
# ============================================================
reviews = ecommerce["review"]

# --- Payments: sum value, count rows, collect distinct payment types used ---
payments_agg = payments.groupby("order_id").agg(
    total_payment_value=("payment_value", "sum"),
    payment_row_count=("order_id", "size"),
    payment_types_used=("payment_type", lambda x: list(x.unique())),
).reset_index()

print(f"\nModule 5: payments_agg shape: {payments_agg.shape} "
      f"(was {payments['order_id'].nunique():,} unique orders before aggregation)")

# --- Reviews: convert date columns first, then keep the LATEST review per order ---
reviews["review_creation_date"] = pd.to_datetime(reviews["review_creation_date"], errors="coerce")
reviews["review_answer_timestamp"] = pd.to_datetime(reviews["review_answer_timestamp"], errors="coerce")

# sort by creation date, then keep the last (most recent) row per order_id
reviews_sorted = reviews.sort_values("review_creation_date")
reviews_agg = reviews_sorted.groupby("order_id").agg(
    review_score=("review_score", "last"),
    review_count=("order_id", "size"),
    has_comment=("review_comment_message", lambda x: x.notna().any()),
).reset_index()

print(f"reviews_agg shape: {reviews_agg.shape} "
      f"(was {reviews['order_id'].nunique():,} unique orders before aggregation)")

ecommerce["payments_agg"] = payments_agg
ecommerce["reviews_agg"] = reviews_agg


# ============================================================
# MODULE 6: Save all cleaned/processed tables to data/processed/
# This closes out Phase 2 — these files are what Phase 3 (EDA) will load.
# ============================================================
os.makedirs("../data/processed", exist_ok=True)

processed_tables = {
    "customers": ecommerce["customers"],
    "orders": ecommerce["orders"],                    # has has_timeline_violation flag
    "order_items": ecommerce["order_items"],
    "products": ecommerce["products"],                 # categories + weights fixed
    "sellers": ecommerce["sellers"],
    "category_translation": ecommerce["category_transalation"],  # 73 rows now
    "geo_clean": geo_clean,                             # from Module 1, 19,010 rows
    "payments_agg": payments_agg,                       # order-level
    "reviews_agg": reviews_agg,                         # order-level
}

for name, df in processed_tables.items():
    path = f"../data/processed/{name}.csv"
    df.to_csv(path, index=False)
    print(f"Saved {name:20s} -> {path}  ({df.shape[0]:,} rows)")
    

Module 4: 'not_defined' rows relabeled: 0 -> 0
Voucher rows with payment_value=0 (kept, valid case): 6

Module 5: payments_agg shape: (99440, 4) (was 99,440 unique orders before aggregation)
reviews_agg shape: (98673, 4) (was 98,673 unique orders before aggregation)
Saved customers            -> ../data/processed/customers.csv  (99,441 rows)
Saved orders               -> ../data/processed/orders.csv  (99,441 rows)
Saved order_items          -> ../data/processed/order_items.csv  (112,650 rows)
Saved products             -> ../data/processed/products.csv  (32,951 rows)
Saved sellers              -> ../data/processed/sellers.csv  (3,095 rows)
Saved category_translation -> ../data/processed/category_translation.csv  (75 rows)
Saved geo_clean            -> ../data/processed/geo_clean.csv  (19,010 rows)
Saved payments_agg         -> ../data/processed/payments_agg.csv  (99,440 rows)
Saved reviews_agg          -> ../data/processed/reviews_agg.csv  (98,673 rows)


In [18]:
category_translation = ecommerce["category_transalation"]

before = len(category_translation)
category_translation = category_translation.drop_duplicates(subset="product_category_name", keep="first")
after = len(category_translation)

print(f"Translation rows: {before} -> {after}")

ecommerce["category_transalation"] = category_translation
category_translation.to_csv("../data/processed/category_translation.csv", index=False)

Translation rows: 73 -> 73
